In [1]:
import collections
import tests.eval as eval
import faiss
import os
import pickle
import re
import retrievers
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import utils

In [2]:
# Variables

TITLES = '.titles.pkl'
CORPUS = '.wiki'
FAISS = '.index.faiss'
QUESTIONS = 'assets/questions.txt'

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 128  # Adjust based on VRAM (GTX 1070 has 8GB)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# FAISS
# Dim 384 for MiniLM-L6. FlatL2 is simple and accurate for this scale.
dimension = 384
index = faiss.IndexFlatL2(dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Load Corpus

if not os.path.exists(FAISS):
    
    wiki_titles = []
    wiki_redirects = collections.defaultdict(lambda: [])
    wiki_content = []

    for title, content in utils.yield_wiki_corpus(CORPUS):
        content = utils.clean_mediawiki(content)
        
        # if alias, track but dont actually index
        found = re.search(r'^\s*#redirect\s+(.*?)$', content, re.IGNORECASE|re.MULTILINE)
        if found:
            alias = found.group(1)
            wiki_redirects[title].append(alias)
            continue
        
        # subdivide into chunks
        for chunk in utils.chunk_text(title, content, window_size=100, overlap=20):
            wiki_titles.append(title)
            wiki_content.append(chunk)

    print('Num Chunks:', len(wiki_content))

In [4]:
# Pickle the mapping: docID -> all title aliases

if not os.path.exists(FAISS):
    
    if os.path.exists(TITLES): os.remove(TITLES)

    with open(TITLES, 'wb') as f: 
        title_mapping = [[title] + wiki_redirects[title] for title in wiki_titles]
        pickle.dump(title_mapping, f)
        
    print(f"Success saving {TITLES}")

In [5]:
# Batch Encoding

if not os.path.exists(FAISS):
    
    def mean_pooling(model_output, attention_mask):
        """Perform mean pooling on token embeddings to get a single vector."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


    for i in tqdm(range(0, len(wiki_content), BATCH_SIZE)):
        batch_text = wiki_content[i : i + BATCH_SIZE]
        
        # Tokenization
        encoded_input = tokenizer(
            batch_text, 
            padding=True, 
            truncation=True, 
            max_length=128, 
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            # Use Automatic Mixed Precision (FP16) for speed boost on GTX 1070
            with torch.cuda.amp.autocast():
                model_output = model(**encoded_input)
                
            # Pool embeddings and move to CPU
            batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
            batch_embeddings = batch_embeddings.cpu().numpy()
            
            # Add to FAISS index immediately to save system RAM
            index.add(batch_embeddings.astype('float32'))

    # Save the index
    faiss.write_index(index, FAISS)
    print(f"Success saving {FAISS}")

In [6]:
# Evaluation

ir = retrievers.JeopardyDPR(FAISS, TITLES, tokenizer, model)
questions = [_ for _ in utils.yield_questions(QUESTIONS)]
k = 100

eval.eval_mrr(ir, questions, k)
eval.eval_p_at_1(ir, questions, k)

MRR@100 (JeopardyDPR): 0.0
P@1 (JeopardyDPR): 0.0
